In [1]:
# file name: display_sensitive.ipynb

# source: file name: R0_ss_sensitive.ipynb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# robust Pearson (same as I gave earlier)
def pearsonr_np(x, y):
    x = np.asarray(x, float).ravel()
    y = np.asarray(y, float).ravel()
    # drop NaN pairs
    m = ~(np.isnan(x) | np.isnan(y))
    x, y = x[m], y[m]
    if x.size < 2:
        return np.nan, x.size
    x = x - x.mean(); y = y - y.mean()
    sx = np.sqrt(np.dot(x, x)); sy = np.sqrt(np.dot(y, y))
    if sx == 0.0 or sy == 0.0:
        return np.nan, x.size
    r = float(np.dot(x, y) / (sx * sy))
    return max(-1.0, min(1.0, r)), x.size

# --- group by (x1, x2) and correlate x3 vs y1 across the x3 repetitions ---
def corr_one_group(g):
    r, n = pearsonr_np(g["Dimmunity"].values, g["avg_time"].values)
    return pd.Series({"r": r, "n": n})

In [3]:
# --- your CSV ---
files = [
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch00.csv",
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch01.csv",
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch02.csv",
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch03.csv",
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch04.csv",
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch05.csv",
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch06.csv",
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch07.csv",
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch08.csv",
    "../../experimental_data/from_260430/R0_sigma_sensitive_2params_batch09.csv"
]

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
cols = ["R0", "sigma", "run_seed"]  # change as needed
print(df[cols].nunique(dropna=True))
df.to_csv("../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv", index=False)
print("Saved merged.csv, rows:", len(df))

R0            12
sigma         10
run_seed    1000
dtype: int64
Saved merged.csv, rows: 120000


In [4]:
def plot_2x5_A2_panels_with_reps_v1(
    csv_path,
    out_dir,
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    rep_col="rep",
    dpi=180,
    connect=True,
    alpha=0.25,
    lw=0.8,
    # --- new ---
    y_mode="robust",          # "robust" or "minmax"
    y_quantiles=(0.01, 0.99), # used if y_mode="robust"
    y_pad=0.05                # extra padding fraction
):
    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_csv(csv_path)

    # Ensure numeric
    for c in [A1_col, A2_col, B1_col, rep_col]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=[A1_col, A2_col, B1_col, rep_col]).copy()

    # Avoid float grouping issues
    df[A2_col] = df[A2_col].round(10)
    df[A1_col] = df[A1_col].round(10)

    A2_vals = np.sort(df[A2_col].unique())
    n_panels = min(len(A2_vals), 10)

    # ---- adaptive y-limits computed from all y values (that will be plotted) ----
    y_all = df[B1_col].to_numpy(dtype=float)
    y_all = y_all[np.isfinite(y_all)]
    if y_all.size == 0:
        raise ValueError(f"No finite values found in {B1_col}.")

    if y_mode == "minmax":
        y0, y1 = float(y_all.min()), float(y_all.max())
    elif y_mode == "robust":
        q0, q1 = y_quantiles
        y0, y1 = np.quantile(y_all, [q0, q1]).astype(float)
    else:
        raise ValueError("y_mode must be 'robust' or 'minmax'")

    # padding
    span = y1 - y0
    if span <= 0:
        span = max(abs(y0), 1.0)
    y0 = y0 - y_pad * span
    y1 = y1 + y_pad * span

    fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=True, sharey=True)
    axes = axes.ravel()

    for k, a2 in enumerate(A2_vals[:n_panels]):
        ax = axes[k]
        g = df[df[A2_col] == a2]

        for rep, gg in g.groupby(rep_col, sort=True):
            gg = gg.sort_values(A1_col)
            x = gg[A1_col].to_numpy()
            y = gg[B1_col].to_numpy()

            if connect:
                ax.plot(x, y, linewidth=lw, alpha=alpha)
            else:
                ax.scatter(x, y, s=10, alpha=alpha)

        ax.set_title(f"{A2_col}={a2:g}", fontsize=11)
        ax.set_ylim(y0, y1)  # apply adaptive limits

    for k in range(n_panels, 10):
        axes[k].axis("off")

    fig.suptitle(f"{A1_col} vs {B1_col} (each line = one seed/run)", fontsize=14)
    fig.supxlabel(A1_col)
    fig.supylabel(B1_col)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out_path = os.path.join(out_dir, f"{A1_col}_vs_{B1_col}_panels_by_{A2_col}.png")
    fig.savefig(out_path, dpi=dpi)
    plt.close(fig)
    print("Saved:", out_path)
    print(f"[Y range] mode={y_mode} -> ({y0:.3g}, {y1:.3g})")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_2x5_A2_panels_with_reps_v2(
    csv_path,
    out_dir,
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    rep_col="run_seed",   # or "rep"
    dpi=180,
    alpha=0.25,
    lw=0.8,
    ms=3.0,
    marker="o",
    # y-range
    y_mode="robust",
    y_quantiles=(0.01, 0.99),
    y_pad=0.05,
    # plotted definition
    min_points_per_rep=4,
    legend_loc="upper left",
):
    os.makedirs(out_dir, exist_ok=True)

    # ========= 1) Read raw =========
    df0 = pd.read_csv(csv_path)

    # ========= 2) TOTAL seeds per A2: only need A2 + rep =========
    df_total = df0.copy()
    for c in [A2_col, rep_col]:
        df_total[c] = pd.to_numeric(df_total[c], errors="coerce")
    df_total = df_total.dropna(subset=[A2_col, rep_col]).copy()
    df_total[A2_col] = df_total[A2_col].round(10)

    # ========= 3) PLOT data: need A1 + A2 + B1 + rep (no NaNs) =========
    df_plot = df0.copy()
    for c in [A1_col, A2_col, B1_col, rep_col]:
        df_plot[c] = pd.to_numeric(df_plot[c], errors="coerce")
    df_plot = df_plot.dropna(subset=[A1_col, A2_col, B1_col, rep_col]).copy()
    df_plot[A2_col] = df_plot[A2_col].round(10)
    df_plot[A1_col] = df_plot[A1_col].round(10)

    # Panels
    A2_vals = np.sort(df_total[A2_col].unique())
    n_panels = min(len(A2_vals), 10)

    # ========= 4) y-limits computed from PLOTTED values only =========
    y_all = df_plot[B1_col].to_numpy(float)
    y_all = y_all[np.isfinite(y_all)]
    if y_all.size == 0:
        raise ValueError(f"No finite values in {B1_col} after cleaning (df_plot).")

    if y_mode == "minmax":
        y0, y1 = float(y_all.min()), float(y_all.max())
    elif y_mode == "robust":
        q0, q1 = y_quantiles
        y0, y1 = np.quantile(y_all, [q0, q1]).astype(float)
    else:
        raise ValueError("y_mode must be 'robust' or 'minmax'")

    span = y1 - y0
    if not np.isfinite(span) or span <= 0:
        span = max(abs(y0), 1.0)
    y0 -= y_pad * span
    y1 += y_pad * span

    # ========= 5) Plot =========
    fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=True, sharey=True)
    axes = axes.ravel()

    for k, a2 in enumerate(A2_vals[:n_panels]):
        ax = axes[k]

        # ----- TOTAL seeds (not affected by NaNs in B1) -----
        total_reps = int(df_total.loc[df_total[A2_col] == a2, rep_col].nunique())

        # ----- PLOTTED seeds (must have enough valid points) -----
        g = df_plot[df_plot[A2_col] == a2]
        plotted_reps = 0

        for rep, gg in g.groupby(rep_col, sort=True):
            # average duplicates at same A1 within a seed
            gg = gg.groupby(A1_col, as_index=False)[B1_col].mean().sort_values(A1_col)

            x = gg[A1_col].to_numpy()
            y = gg[B1_col].to_numpy()
            m = np.isfinite(x) & np.isfinite(y)

            if m.sum() < min_points_per_rep:
                continue

            plotted_reps += 1
            ax.plot(
                x[m], y[m],
                linestyle="-",
                linewidth=lw,
                alpha=alpha,
                marker=marker,
                markersize=ms,
                markerfacecolor="none",
            )

        ax.set_title(f"{A2_col}={a2:g}", fontsize=11)
        ax.set_ylim(y0, y1)

        # legend text (total vs plotted)
        txt = f"samples: {total_reps}\nvalid: {plotted_reps}"
        ax.plot([], [], " ", label=txt)  # invisible handle
        ax.legend(loc=legend_loc, frameon=True, fontsize=9, handlelength=0, handletextpad=0)

    # hide unused axes
    for k in range(n_panels, 10):
        axes[k].axis("off")

    fig.suptitle(f"{A1_col} vs {B1_col} (line+dot per {rep_col})", fontsize=14)
    fig.supxlabel(A1_col)
    fig.supylabel(B1_col)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out_path = os.path.join(out_dir, f"{A1_col}_vs_{B1_col}_panels_by_{A2_col}.png")
    fig.savefig(out_path, dpi=dpi)
    plt.close(fig)

    print("Saved:", out_path)
    print(f"[Y range] mode={y_mode} -> ({y0:.3g}, {y1:.3g})")

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_2x5_A2_panels_band_only_old(
    csv_path,
    out_dir,
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    rep_col="rep",
    dpi=180,
    center="mean",           # "mean" or "median"
    band=(0.05, 0.95),       # band within each panel
    min_n=30,                # ignore A1 positions with < min_n values
    # ---- y-range control (match plot_2x5_A2_panels_with_reps_v2 style) ----
    y_mode="robust",         # "robust" or "minmax"
    y_quantiles=(0.05, 0.99),
    top_pad=0.25,
    bottom_pad=0.05,
):
    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_csv(csv_path)

    # numeric safety
    for c in [A1_col, A2_col, B1_col, rep_col]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=[A1_col, A2_col, B1_col, rep_col]).copy()

    df[A2_col] = df[A2_col].round(10)
    df[A1_col] = df[A1_col].round(10)

    A2_vals = np.sort(df[A2_col].unique())
    n_panels = min(len(A2_vals), 10)
    qlo, qhi = band

    # ---- compute y-limits from ALL B1 values (same idea as v2) ----
    y_all = df[B1_col].to_numpy(dtype=float)
    y_all = y_all[np.isfinite(y_all)]
    if y_all.size == 0:
        raise ValueError(f"No finite values found in {B1_col}.")

    if y_mode == "minmax":
        y0, y1 = float(y_all.min()), float(y_all.max())
    elif y_mode == "robust":
        q0, q1 = y_quantiles
        y0, y1 = np.quantile(y_all, [q0, q1]).astype(float)
    else:
        raise ValueError("y_mode must be 'robust' or 'minmax'")

    span = y1 - y0
    if not np.isfinite(span) or span <= 0:
        span = max(abs(y0), 1.0)
    y0 = y0 - bottom_pad * span
    y1 = y1 + top_pad * span

    # ---- plot ----
    fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=True, sharey=True)
    axes = axes.ravel()

    for k, a2 in enumerate(A2_vals[:n_panels]):
        ax = axes[k]
        g = df[df[A2_col] == a2]
        if g.empty:
            ax.axis("off")
            continue

        P = g.pivot_table(index=rep_col, columns=A1_col, values=B1_col, aggfunc="mean").sort_index(axis=1)
        x = P.columns.to_numpy(float)
        Y = P.to_numpy(float)

        # support masking to avoid “false certainty” from sparse values
        n_valid = np.sum(np.isfinite(Y), axis=0)
        keep = n_valid >= min_n

        if center == "median":
            c = np.nanmedian(Y, axis=0)
            c_label = "Median"
        else:
            c = np.nanmean(Y, axis=0)
            c_label = "Mean"

        lo = np.nanquantile(Y, qlo, axis=0)
        hi = np.nanquantile(Y, qhi, axis=0)

        m = keep & np.isfinite(c) & np.isfinite(lo) & np.isfinite(hi)
        if m.sum() >= 2:
            ax.fill_between(x[m], lo[m], hi[m], alpha=0.25,
                            label=f"{int(qlo*100)}–{int(qhi*100)}% band (n≥{min_n})")
            ax.plot(x[m], c[m], linewidth=2.2, label=c_label)

        ax.set_title(f"{A2_col}={a2:g}", fontsize=11)
        ax.set_ylim(y0, y1)

    for k in range(n_panels, 10):
        axes[k].axis("off")

    fig.suptitle(
        f"{A1_col} vs {B1_col}: {center} + {int(qlo*100)}–{int(qhi*100)}% band (per {A2_col})",
        fontsize=14
    )
    fig.supxlabel(A1_col)
    fig.supylabel(B1_col)

    # one legend for whole figure
    for ax in axes[:n_panels]:
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc="upper right", frameon=False)
            break

    fig.tight_layout(rect=[0, 0, 1, 0.94])

    out_path = os.path.join(out_dir, f"{A1_col}_vs_{B1_col}_{center}_band_by_{A2_col}.png")
    fig.savefig(out_path, dpi=dpi)
    plt.close(fig)
    print("Saved:", out_path)
    print(f"[Y range] mode={y_mode}, y_quantiles={y_quantiles} -> ({y0:.3g}, {y1:.3g})")



def plot_2x5_A2_panels_band_only_v1(
    csv_path,
    out_dir,
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    rep_col="run_seed",
    dpi=180,
    center="mean",             # "mean" or "median"
    band=(0.05, 0.95),
    min_n=30,                  # ignore A1 positions with < min_n values
    min_points_per_rep=4,      # valid: seed must have >= this many points (in kept columns)
    y_mode="robust",
    y_quantiles=(0.05, 0.99),
    top_pad=0.25,
    bottom_pad=0.05,
    legend_loc="upper left",
):
    os.makedirs(out_dir, exist_ok=True)

    df0 = pd.read_csv(csv_path)

    # ----------------------------
    # 1) samples df: count seeds BEFORE dropping B1 NaNs
    # ----------------------------
    df_samples = df0.copy()
    for c in [A2_col, rep_col]:
        df_samples[c] = pd.to_numeric(df_samples[c], errors="coerce")
    df_samples = df_samples.dropna(subset=[A2_col, rep_col]).copy()
    df_samples[A2_col] = df_samples[A2_col].round(10)

    # ----------------------------
    # 2) plot df: remove NaNs needed for band/center calculation
    # ----------------------------
    df_plot = df0.copy()
    for c in [A1_col, A2_col, B1_col, rep_col]:
        df_plot[c] = pd.to_numeric(df_plot[c], errors="coerce")
    df_plot = df_plot.dropna(subset=[A1_col, A2_col, B1_col, rep_col]).copy()
    df_plot[A2_col] = df_plot[A2_col].round(10)
    df_plot[A1_col] = df_plot[A1_col].round(10)

    A2_vals = np.sort(df_samples[A2_col].unique())
    n_panels = min(len(A2_vals), 10)
    qlo, qhi = band

    # ---- y-limits computed from PLOTTED values ----
    y_all = df_plot[B1_col].to_numpy(float)
    y_all = y_all[np.isfinite(y_all)]
    if y_all.size == 0:
        raise ValueError(f"No finite values found in {B1_col} (after cleaning df_plot).")

    if y_mode == "minmax":
        y0, y1 = float(y_all.min()), float(y_all.max())
    elif y_mode == "robust":
        q0, q1 = y_quantiles
        y0, y1 = np.quantile(y_all, [q0, q1]).astype(float)
    else:
        raise ValueError("y_mode must be 'robust' or 'minmax'")

    span = y1 - y0
    if not np.isfinite(span) or span <= 0:
        span = max(abs(y0), 1.0)
    y0 -= bottom_pad * span
    y1 += top_pad * span

    fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=True, sharey=True)
    axes = axes.ravel()

    # figure-level legend handles (band + center)
    band_handle = None
    center_handle = None
    center_label = "Median" if center == "median" else "Mean"
    band_label = f"{int(qlo*100)}–{int(qhi*100)}% band (n≥{min_n})"

    for k, a2 in enumerate(A2_vals[:n_panels]):
        ax = axes[k]

        # samples: total unique run_seed for this A2 (before dropping B1 NaNs)
        samples = int(df_samples.loc[df_samples[A2_col] == a2, rep_col].nunique())

        g = df_plot[df_plot[A2_col] == a2]
        if g.empty:
            ax.set_title(f"{A2_col}={a2:g}", fontsize=11)
            ax.set_ylim(y0, y1)
            txt = f"samples: {samples}\nvalid: 0"
            ax.plot([], [], " ", label=txt)
            ax.legend(loc=legend_loc, frameon=True, fontsize=9, handlelength=0, handletextpad=0)
            continue

        # reps × A1 pivot
        P = g.pivot_table(index=rep_col, columns=A1_col, values=B1_col, aggfunc="mean").sort_index(axis=1)
        x = P.columns.to_numpy(float)
        Y = P.to_numpy(float)  # (n_seeds_with_data, n_A1)

        # keep A1 positions with enough observations across seeds
        n_valid_col = np.sum(np.isfinite(Y), axis=0)
        keep = n_valid_col >= min_n

        # valid seeds: must have >= min_points_per_rep finite points in kept columns
        if np.any(keep):
            Yk = Y[:, keep]
            counts_per_seed = np.sum(np.isfinite(Yk), axis=1)
            valid = int(np.sum(counts_per_seed >= min_points_per_rep))
        else:
            valid = 0

        # center + band statistics (computed on Y; displayed only on kept columns)
        if center == "median":
            c = np.nanmedian(Y, axis=0)
        else:
            c = np.nanmean(Y, axis=0)
        lo = np.nanquantile(Y, qlo, axis=0)
        hi = np.nanquantile(Y, qhi, axis=0)

        m = keep & np.isfinite(c) & np.isfinite(lo) & np.isfinite(hi)
        if m.sum() >= 2:
            fb = ax.fill_between(x[m], lo[m], hi[m], alpha=0.25)
            ln, = ax.plot(x[m], c[m], linewidth=2.2)
            if band_handle is None:
                band_handle = fb
            if center_handle is None:
                center_handle = ln

        ax.set_title(f"{A2_col}={a2:g}", fontsize=11)
        ax.set_ylim(y0, y1)

        # subplot legend: ONLY samples/valid
        txt = f"samples: {samples}\nvalid: {valid}"
        ax.plot([], [], " ", label=txt)
        ax.legend(loc=legend_loc, frameon=True, fontsize=9, handlelength=0, handletextpad=0)

    for k in range(n_panels, 10):
        axes[k].axis("off")

    fig.suptitle(
        f"{A1_col} vs {B1_col}: {center_label} + {int(qlo*100)}–{int(qhi*100)}% band (per {A2_col})",
        fontsize=14
    )
    fig.supxlabel(A1_col)
    fig.supylabel(B1_col)

    # figure-level legend for band + center
    if band_handle is not None and center_handle is not None:
        fig.legend([center_handle, band_handle], [center_label, band_label],
                   loc="upper right", frameon=False)

    fig.tight_layout(rect=[0, 0, 1, 0.94])

    out_path = os.path.join(out_dir, f"{A1_col}_vs_{B1_col}_{center}_band_by_{A2_col}_samples_valid.png")
    fig.savefig(out_path, dpi=dpi)
    plt.close(fig)

    print("Saved:", out_path)
    print(f"[Y range] mode={y_mode}, y_quantiles={y_quantiles} -> ({y0:.3g}, {y1:.3g})")


def plot_2x5_A2_panels_band_only(
    csv_path,
    out_dir,
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    rep_col="run_seed",
    dpi=180,
    center="mean",               # "mean" or "median"
    band=(0.05, 0.95),
    min_n=30,                    # keep A1 columns with >= min_n valid seeds contributing
    min_points_per_rep=4,        # seed validity: must have >= this many A1 points
    y_mode="robust",
    y_quantiles=(0.05, 0.99),
    top_pad=0.25,
    bottom_pad=0.05,
    legend_loc="upper left",
):
    os.makedirs(out_dir, exist_ok=True)
    df0 = pd.read_csv(csv_path)

    # ---- samples (count seeds BEFORE dropping B1 NaNs) ----
    df_samples = df0.copy()
    for c in [A2_col, rep_col]:
        df_samples[c] = pd.to_numeric(df_samples[c], errors="coerce")
    df_samples = df_samples.dropna(subset=[A2_col, rep_col]).copy()
    df_samples[A2_col] = df_samples[A2_col].round(10)

    # ---- plot data (need A1/A2/B1/seed) ----
    df_plot = df0.copy()
    for c in [A1_col, A2_col, B1_col, rep_col]:
        df_plot[c] = pd.to_numeric(df_plot[c], errors="coerce")
    df_plot = df_plot.dropna(subset=[A1_col, A2_col, B1_col, rep_col]).copy()
    df_plot[A2_col] = df_plot[A2_col].round(10)
    df_plot[A1_col] = df_plot[A1_col].round(10)

    A2_vals = np.sort(df_samples[A2_col].unique())
    n_panels = min(len(A2_vals), 10)
    qlo, qhi = band

    # ---- y-limits from plotted values ----
    y_all = df_plot[B1_col].to_numpy(float)
    y_all = y_all[np.isfinite(y_all)]
    if y_all.size == 0:
        raise ValueError(f"No finite values found in {B1_col} (after cleaning df_plot).")

    if y_mode == "minmax":
        y0, y1 = float(y_all.min()), float(y_all.max())
    elif y_mode == "robust":
        q0, q1 = y_quantiles
        y0, y1 = np.quantile(y_all, [q0, q1]).astype(float)
    else:
        raise ValueError("y_mode must be 'robust' or 'minmax'")

    span = y1 - y0
    if not np.isfinite(span) or span <= 0:
        span = max(abs(y0), 1.0)
    y0 -= bottom_pad * span
    y1 += top_pad * span

    fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=True, sharey=True)
    axes = axes.ravel()

    # figure-level legend handles (band + center)
    band_handle = None
    center_handle = None
    center_label = "Median" if center == "median" else "Mean"
    band_label = f"{int(qlo*100)}–{int(qhi*100)}% band"

    for k, a2 in enumerate(A2_vals[:n_panels]):
        ax = axes[k]

        samples = int(df_samples.loc[df_samples[A2_col] == a2, rep_col].nunique())

        g = df_plot[df_plot[A2_col] == a2]
        if g.empty:
            ax.set_title(f"{A2_col}={a2:g}", fontsize=11)
            ax.set_ylim(y0, y1)
            txt = f"samples: {samples}\nvalid: 0"
            ax.plot([], [], " ", label=txt)
            ax.legend(loc=legend_loc, frameon=True, fontsize=9, handlelength=0, handletextpad=0)
            continue

        # reps × A1 pivot
        P = g.pivot_table(index=rep_col, columns=A1_col, values=B1_col, aggfunc="mean").sort_index(axis=1)
        x = P.columns.to_numpy(float)
        Y = P.to_numpy(float)  # (n_seeds_with_any_data, n_A1)

        # ---- seed validity like with_reps_v2: >= min_points_per_rep points anywhere ----
        counts_per_seed = np.sum(np.isfinite(Y), axis=1)
        valid_seed_mask = counts_per_seed >= min_points_per_rep
        valid = int(valid_seed_mask.sum())

        if valid < 2:
            # too few valid seeds to compute quantiles meaningfully
            ax.set_title(f"{A2_col}={a2:g}", fontsize=11)
            ax.set_ylim(y0, y1)
            txt = f"samples: {samples}\nvalid: {valid}"
            ax.plot([], [], " ", label=txt)
            ax.legend(loc=legend_loc, frameon=True, fontsize=9, handlelength=0, handletextpad=0)
            continue

        # Filter to valid seeds only for band/center
        Yv = Y[valid_seed_mask, :]

        # ---- keep columns based on valid seeds only (optional but consistent) ----
        n_valid_col = np.sum(np.isfinite(Yv), axis=0)
        keep = n_valid_col >= min_n

        # center + band computed from valid seeds
        if center == "median":
            c = np.nanmedian(Yv, axis=0)
        else:
            c = np.nanmean(Yv, axis=0)

        lo = np.nanquantile(Yv, qlo, axis=0)
        hi = np.nanquantile(Yv, qhi, axis=0)

        m = keep & np.isfinite(c) & np.isfinite(lo) & np.isfinite(hi)
        if m.sum() >= 2:
            fb = ax.fill_between(x[m], lo[m], hi[m], alpha=0.25)
            ln, = ax.plot(x[m], c[m], linewidth=2.2)
            if band_handle is None:
                band_handle = fb
            if center_handle is None:
                center_handle = ln

        ax.set_title(f"{A2_col}={a2:g}", fontsize=11)
        ax.set_ylim(y0, y1)

        # subplot legend: samples/valid only
        txt = f"samples: {samples}\nvalid: {valid}"
        ax.plot([], [], " ", label=txt)
        ax.legend(loc=legend_loc, frameon=True, fontsize=9, handlelength=0, handletextpad=0)

    for k in range(n_panels, 10):
        axes[k].axis("off")

    fig.suptitle(
        f"{A1_col} vs {B1_col}: {center_label} + {int(qlo*100)}–{int(qhi*100)}% band (per {A2_col})",
        fontsize=14
    )
    fig.supxlabel(A1_col)
    fig.supylabel(B1_col)

    # figure-level legend for band + center
    if band_handle is not None and center_handle is not None:
        fig.legend([center_handle, band_handle], [center_label, band_label],
                   loc="upper right", frameon=False)

    fig.tight_layout(rect=[0, 0, 1, 0.94])

    out_path = os.path.join(out_dir, f"{A1_col}_vs_{B1_col}_{center}_band_by_{A2_col}.png")
    fig.savefig(out_path, dpi=dpi)
    plt.close(fig)

    print("Saved:", out_path)
    print(f"[Y range] mode={y_mode}, y_quantiles={y_quantiles} -> ({y0:.3g}, {y1:.3g})")
    

In [6]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_time_panels_by_sigma.png
[Y range] mode=robust -> (-2.95, 26.5)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_time_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.95, 26.5)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_time_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.95, 26.5)


In [7]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="max_time",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="max_time",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="max_time",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_max_time_panels_by_sigma.png
[Y range] mode=robust -> (-2.2, 27.2)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_max_time_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.2, 27.2)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_max_time_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.2, 27.2)


In [8]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="num_strains",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="num_strains",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="num_strains",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_num_strains_panels_by_sigma.png
[Y range] mode=robust -> (-4.2, 39.2)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_num_strains_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-4.2, 39.2)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_num_strains_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-4.2, 39.2)


In [9]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time_repeat",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time_repeat",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time_repeat",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_time_repeat_panels_by_sigma.png
[Y range] mode=robust -> (0.8, 2.2)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_time_repeat_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (0.8, 2.2)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_time_repeat_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (0.8, 2.2)


In [10]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="var_time_repeat",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="var_time_repeat",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="var_time_repeat",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_var_time_repeat_panels_by_sigma.png
[Y range] mode=robust -> (-1.31, 7.89)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_var_time_repeat_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-1.31, 7.89)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_var_time_repeat_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-1.31, 7.89)


In [11]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_prev",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_prev",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_prev",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_prev_panels_by_sigma.png
[Y range] mode=robust -> (-85.8, 516)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_prev_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-85.8, 516)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_prev_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-85.8, 516)


In [12]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="var_prev",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="var_prev",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="var_prev",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_var_prev_panels_by_sigma.png
[Y range] mode=robust -> (-9.33e+03, 5.6e+04)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_var_prev_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-9.33e+03, 5.6e+04)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_var_prev_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-9.33e+03, 5.6e+04)


In [13]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_div",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_div",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_div",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_div_panels_by_sigma.png
[Y range] mode=robust -> (-5.7, 35.4)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_div_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-5.7, 35.4)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_div_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-5.7, 35.4)


In [14]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="var_div",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="var_div",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="var_div",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_var_div_panels_by_sigma.png
[Y range] mode=robust -> (-2.17, 13.8)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_var_div_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.17, 13.8)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_var_div_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.17, 13.8)


In [15]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="max_abundance",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="max_abundance",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="max_abundance",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_max_abundance_panels_by_sigma.png
[Y range] mode=robust -> (-47.4, 291)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_max_abundance_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-47.4, 291)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_max_abundance_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-47.4, 291)


In [16]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_npmi",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_npmi",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_npmi",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_npmi_panels_by_sigma.png
[Y range] mode=robust -> (-0.763, 0.377)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_npmi_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-0.763, 0.377)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_avg_npmi_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-0.763, 0.377)


In [17]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="div_all_isolates",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="div_all_isolates",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/R0_vs_ss_panels_by_sigma",
    A1_col="R0",
    A2_col="sigma",
    B1_col="div_all_isolates",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_div_all_isolates_panels_by_sigma.png
[Y range] mode=robust -> (-4.21, 35)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_div_all_isolates_mean_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-4.21, 35)
Saved: ../../figures/from_260312/R0_vs_ss_panels_by_sigma/R0_vs_div_all_isolates_median_band_by_sigma.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-4.21, 35)


In [18]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_time",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_time",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_time",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_time_panels_by_R0.png
[Y range] mode=robust -> (-2.95, 26.5)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_time_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.95, 26.5)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_time_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.95, 26.5)


In [19]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="max_time",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="max_time",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="max_time",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_max_time_panels_by_R0.png
[Y range] mode=robust -> (-2.2, 27.2)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_max_time_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.2, 27.2)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_max_time_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.2, 27.2)


In [20]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="num_strains",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="num_strains",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="num_strains",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_num_strains_panels_by_R0.png
[Y range] mode=robust -> (-4.2, 39.2)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_num_strains_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-4.2, 39.2)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_num_strains_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-4.2, 39.2)


In [21]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_time_repeat",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_time_repeat",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_time_repeat",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_time_repeat_panels_by_R0.png
[Y range] mode=robust -> (0.8, 2.2)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_time_repeat_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (0.8, 2.2)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_time_repeat_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (0.8, 2.2)


In [22]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="var_time_repeat",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="var_time_repeat",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="var_time_repeat",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_var_time_repeat_panels_by_R0.png
[Y range] mode=robust -> (-1.31, 7.89)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_var_time_repeat_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-1.31, 7.89)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_var_time_repeat_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-1.31, 7.89)


In [23]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_prev",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_prev",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_prev",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_prev_panels_by_R0.png
[Y range] mode=robust -> (-85.8, 516)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_prev_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-85.8, 516)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_prev_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-85.8, 516)


In [24]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="var_prev",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="var_prev",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="var_prev",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_var_prev_panels_by_R0.png
[Y range] mode=robust -> (-9.33e+03, 5.6e+04)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_var_prev_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-9.33e+03, 5.6e+04)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_var_prev_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-9.33e+03, 5.6e+04)


In [25]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_div",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_div",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_div",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_div_panels_by_R0.png
[Y range] mode=robust -> (-5.7, 35.4)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_div_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-5.7, 35.4)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_div_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-5.7, 35.4)


In [26]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="var_div",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="var_div",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="var_div",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_var_div_panels_by_R0.png
[Y range] mode=robust -> (-2.17, 13.8)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_var_div_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.17, 13.8)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_var_div_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-2.17, 13.8)


In [27]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="max_abundance",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="max_abundance",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="max_abundance",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_max_abundance_panels_by_R0.png
[Y range] mode=robust -> (-47.4, 291)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_max_abundance_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-47.4, 291)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_max_abundance_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-47.4, 291)


In [28]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_npmi",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_npmi",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="avg_npmi",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_npmi_panels_by_R0.png
[Y range] mode=robust -> (-0.763, 0.377)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_npmi_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-0.763, 0.377)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_avg_npmi_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-0.763, 0.377)


In [29]:
plot_2x5_A2_panels_with_reps_v2(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="div_all_isolates",
    rep_col="run_seed",   # or "run_seed" if that's what you saved
    y_mode="robust",
    y_quantiles=(0.01, 0.99), 
    y_pad=0.20
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="div_all_isolates",
    rep_col="run_seed",
    min_n=300,
    center="mean",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
)

plot_2x5_A2_panels_band_only(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    out_dir="../../figures/from_260430/sigma_vs_ss_panels_by_R0",
    A1_col="sigma",
    A2_col="R0",
    B1_col="div_all_isolates",
    rep_col="run_seed",
    min_n=300,
    center="median",
    top_pad=0.20,    # more headroom
    bottom_pad=0.20,
    y_quantiles=(0.01, 0.99)
   
)

Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_div_all_isolates_panels_by_R0.png
[Y range] mode=robust -> (-4.21, 35)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_div_all_isolates_mean_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-4.21, 35)
Saved: ../../figures/from_260312/sigma_vs_ss_panels_by_R0/sigma_vs_div_all_isolates_median_band_by_R0.png
[Y range] mode=robust, y_quantiles=(0.01, 0.99) -> (-4.21, 35)


In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def pearsonr_np(x, y):
    """Return Pearson r and n used (after dropping NaNs)."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    n = x.size
    if n < 2:
        return np.nan, n
    x = x - x.mean()
    y = y - y.mean()
    denom = np.sqrt((x*x).sum() * (y*y).sum())
    if denom == 0:
        return np.nan, n
    r = float((x*y).sum() / denom)
    return r, n


def pcc_A1_B1_by_A2_and_seed(
    csv_path,
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    seed_col="run_seed",     # or "rep"
    min_points=5,            # require at least this many A1 points per group
    round_A2=10,             # avoid float grouping issues
    round_A1=10,
):
    df = pd.read_csv(csv_path)

    # numeric safety
    for c in [A1_col, A2_col, B1_col, seed_col]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=[A1_col, A2_col, B1_col, seed_col]).copy()

    # avoid float mismatches (0.3000000004 etc.)
    df[A2_col] = df[A2_col].round(round_A2)
    df[A1_col] = df[A1_col].round(round_A1)

    rows = []
    # group by (A2 fixed, seed fixed)
    for (a2, seed), g in df.groupby([A2_col, seed_col], sort=True):
        g = g.sort_values(A1_col)

        # If you may have duplicates for same (A1) within a seed, average them
        gg = g.groupby(A1_col, as_index=False)[B1_col].mean()

        r, n = pearsonr_np(gg[A1_col].to_numpy(), gg[B1_col].to_numpy())
        if n >= min_points:
            rows.append((a2, seed, r, n))

    res = pd.DataFrame(rows, columns=[A2_col, seed_col, "pcc", "n"])
    return res


def boxplot_pcc_by_A2(
    res_df,
    A2_col="sigma",
    pcc_col="pcc",
    A1_col="R0",
    B1_col="avg_time",
    title=None,
    dpi=180,
    save_path=None,   # None OR folder OR full .png path
    show=False        # if True, show even when saving
):
    # order A2 values
    a2_vals = np.sort(res_df[A2_col].unique())
    
    data = []
    x_labels = []
    count_labels = []
    
    for a2 in a2_vals:
        subset = res_df.loc[res_df[A2_col] == a2, pcc_col]
        n_total = len(subset)
        valid_data = subset.dropna().to_numpy()
        n_valid = len(valid_data)
        
        data.append(valid_data)
        x_labels.append(f"{a2:g}")
        count_labels.append(f"n={n_total}, v={n_valid}")
    
    plt.figure(figsize=(12, 4.5))
    bp = plt.boxplot(data, labels=x_labels, showfliers=False)
    plt.axhline(0.0, linewidth=1)
    
    # Add count labels on top of each box
    ax = plt.gca()
    y_max = ax.get_ylim()[1]
    
    for i, label_text in enumerate(count_labels):
        # Get the maximum whisker height for this box
        if len(data[i]) > 0:
            box_max = bp['whiskers'][i*2 + 1].get_ydata()[1]
            y_pos = max(box_max, y_max * 0.95)
        else:
            y_pos = y_max * 0.95
        
        plt.text(i + 1, y_pos, label_text, 
                ha='center', va='bottom', fontsize=8)
    
    plt.xlabel(A2_col)
    plt.ylabel(f"Pearson r ({A1_col} vs {B1_col})")
    
    if title is None:
        plt.title(f"PCC between {A1_col} and {B1_col} (grouped by {A2_col})")
    else:
        plt.title(title)
    
    plt.tight_layout()
    
    # ----- adaptive save behavior -----
    if save_path is None:
        plt.show()
        return None
    
    # folder vs file path
    save_path = str(save_path)
    if save_path.endswith(os.sep) or os.path.isdir(save_path) or (not save_path.lower().endswith(".png")):
        out_dir = save_path
        os.makedirs(out_dir, exist_ok=True)
        fname = f"pcc_boxplot_{A1_col}_vs_{B1_col}_by_{A2_col}.png".replace(" ", "_")
        out_file = os.path.join(out_dir, fname)
    else:
        out_dir = os.path.dirname(save_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        out_file = save_path
    
    plt.savefig(out_file, dpi=dpi)
    if show:
        plt.show()
    plt.close()
    print("Saved:", out_file)
    return out_file

In [45]:
# 1) compute PCCs
res = pcc_A1_B1_by_A2_and_seed(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    A1_col="R0",
    A2_col="sigma",
    B1_col="avg_time",
    seed_col="run_seed",   # change to "rep" if that’s what you have
    min_points=5
)

# (optional) inspect
print(res.head())
print(res.groupby("sigma")["pcc"].count())  # how many seeds per sigma




   sigma    run_seed       pcc   n
0    0.1    704072.0  0.840219  12
1    0.1   2639917.0  0.851037  12
2    0.1   7184979.0  0.698470  12
3    0.1   8076639.0  0.635701  12
4    0.1  11547993.0  0.655447  12
sigma
0.1    1000
0.2    1000
0.3    1000
0.4    1000
0.5    1000
0.6    1000
0.7    1000
0.8    1000
0.9    1000
1.0    1000
Name: pcc, dtype: int64


In [46]:
boxplot_pcc_by_A2(res, A1_col="R0", B1_col="avg_time", A2_col="sigma", save_path="../../figures/from_260430/")

Saved: ../../figures/from_260312/pcc_boxplot_R0_vs_avg_time_by_sigma.png


'../../figures/from_260312/pcc_boxplot_R0_vs_avg_time_by_sigma.png'

In [48]:
# Check the overall valid count
print("Total rows in res_df:", len(res))
print("Total valid PCC:", res[pcc_col].notna().sum())
print("Total finite PCC:", np.isfinite(res[pcc_col]).sum())

# Check by group
print("\nBy group:")
for a2 in np.sort(res_df[A2_col].unique()):
    subset = res_df.loc[res_df[A2_col] == a2, pcc_col]
    print(f"{A2_col}={a2}: total={len(subset)}, valid={subset.notna().sum()}, finite={np.isfinite(subset).sum()}")

# Check for the missing 28 values
print("\nRows with invalid PCC:")
invalid_pcc = res_df[~np.isfinite(res_df[pcc_col])]
print(invalid_pcc[[A2_col, pcc_col]])

Total rows in res_df: 10000


NameError: name 'pcc_col' is not defined

In [33]:
def violin_pcc_by_A2(
    res_df,
    A2_col="sigma",
    pcc_col="pcc",
    A1_col="R0",
    B1_col="avg_time",
    title=None,
    show_median=True,
    show_iqr=True,
    dpi=180,
    save_path=None,   # None OR folder OR full .png path
    show=False        # if True, show even when saving
):
    a2_vals = np.sort(res_df[A2_col].unique())
    data = [res_df.loc[res_df[A2_col] == a2, pcc_col].dropna().to_numpy()
            for a2 in a2_vals]

    positions = np.arange(1, len(a2_vals) + 1)

    plt.figure(figsize=(12, 4.5))
    plt.violinplot(
        data,
        positions=positions,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    # Add median / IQR overlays
    for i, y in enumerate(data, start=1):
        if y.size == 0:
            continue
        if show_iqr:
            q1, q3 = np.quantile(y, [0.25, 0.75])
            plt.plot([i, i], [q1, q3], linewidth=3)
        if show_median:
            med = np.median(y)
            plt.scatter([i], [med], s=20, zorder=3)

    plt.axhline(0.0, linewidth=1)
    plt.xticks(positions, [f"{v:g}" for v in a2_vals])
    plt.xlabel(A2_col)
    plt.ylabel(f"Pearson r ({A1_col} vs {B1_col})")

    if title is None:
        plt.title(f"PCC between {A1_col} and {B1_col} (grouped by {A2_col})")
    else:
        plt.title(title)

    plt.tight_layout()

    # ----- adaptive save behavior -----
    if save_path is None:
        plt.show()
        return None

    save_path = str(save_path)
    if save_path.endswith(os.sep) or os.path.isdir(save_path) or (not save_path.lower().endswith(".png")):
        out_dir = save_path
        os.makedirs(out_dir, exist_ok=True)
        fname = f"pcc_violin_{A1_col}_vs_{B1_col}_by_{A2_col}.png".replace(" ", "_")
        out_file = os.path.join(out_dir, fname)
    else:
        out_dir = os.path.dirname(save_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        out_file = save_path

    plt.savefig(out_file, dpi=dpi)
    if show:
        plt.show()
    plt.close()
    print("Saved:", out_file)
    return out_file

In [34]:
violin_pcc_by_A2(
    res_df=res,
    A1_col="R0",
    B1_col="avg_time",
    A2_col="sigma",
    save_path="../../figures/from_260430/"   # folder -> auto filename
)

Saved: ../../figures/from_260312/pcc_violin_R0_vs_avg_time_by_sigma.png


'../../figures/from_260312/pcc_violin_R0_vs_avg_time_by_sigma.png'

In [38]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity

def kde_all_pcc(
    res_df,
    pcc_col="pcc",
    A1_col="R0",
    B1_col="avg_time",
    bandwidth=0.08,
    gridsize=800,
    dpi=180,
    save_path=None,          # None OR folder OR full .png path
):
    x = res_df[pcc_col].to_numpy(dtype=float)
    x_valid = x[np.isfinite(x)]
    n_total = x.size
    n_valid = x_valid.size
    
    if n_valid < 2:
        raise ValueError("Not enough valid PCC points for KDE.")
    
    X = x_valid.reshape(-1, 1)
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth).fit(X)
    xs = np.linspace(x_valid.min(), x_valid.max(), gridsize).reshape(-1, 1)
    dens = np.exp(kde.score_samples(xs))
    
    # summary stats
    x_mean = float(np.mean(x_valid))
    x_med  = float(np.median(x_valid))
    x_mode = float(xs[int(np.argmax(dens)), 0])
    
    plt.figure(figsize=(6.6, 4.3))
    plt.hist(x_valid, bins=30, density=True, alpha=0.30, edgecolor="black",
             label=f"Samples (n={n_total})")
    
    # Add "Valid" as separate legend entry
    plt.plot([], [], ' ', label=f"Valid (n={n_valid})")
    
    plt.plot(xs.ravel(), dens, linewidth=2.2, color="C1", label=f"KDE (bw={bandwidth:g})")
    plt.axvline(x_mode, linestyle="-", color="C1", linewidth=1.6, label=f"KDE mode={x_mode:.3g}")
    plt.axvline(x_mean, linestyle="--", linewidth=1.4, label=f"Mean={x_mean:.3g}")
    plt.axvline(x_med,  linestyle=":",  linewidth=1.6, label=f"Median={x_med:.3g}")
    
    plt.xlabel("PCC")
    plt.ylabel("Density")
    plt.title(f"PCC between {A1_col} and {B1_col} (all seeds)")
    plt.legend(loc="center left", fontsize=8, bbox_to_anchor=(0.72, 0.78), borderaxespad=0.)
    plt.tight_layout()
    
    # --- adaptive save behavior ---
    if save_path is None:
        plt.show()
        return None
    
    # If save_path is a directory (or has no .png), treat as folder
    if save_path.endswith(os.sep) or (os.path.isdir(save_path)) or (not str(save_path).lower().endswith(".png")):
        out_dir = save_path
        os.makedirs(out_dir, exist_ok=True)
        fname = f"pcc_kde_all_{A1_col}_vs_{B1_col}.png".replace(" ", "_")
        out_file = os.path.join(out_dir, fname)
    else:
        # full file path provided
        out_dir = os.path.dirname(save_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        out_file = save_path
    
    plt.savefig(out_file, dpi=dpi)
    plt.close()
    print("Saved:", out_file)
    return out_file

In [39]:
kde_all_pcc(res, A1_col="R0", B1_col="avg_time", save_path="../../figures/from_260430/")

Saved: ../../figures/from_260312/pcc_kde_all_R0_vs_avg_time.png


'../../figures/from_260312/pcc_kde_all_R0_vs_avg_time.png'

In [40]:
# 1) compute PCCs
A1_col="R0"
A2_col="sigma"
B1_col="max_time"

res = pcc_A1_B1_by_A2_and_seed(
    csv_path="../../experimental_data/from_260430/R0_sigma_sensitive_2params_merged.csv",
    A1_col=A1_col,
    A2_col=A2_col,
    B1_col=B1_col,
    seed_col="run_seed",   # change to "rep" if that’s what you have
    min_points=5
)

boxplot_pcc_by_A2(res, 
                  A1_col=A1_col,
                  A2_col=A2_col,
                  B1_col=B1_col,
                  save_path="../../figures/from_260430/")

violin_pcc_by_A2(res, 
                  A1_col=A1_col,
                  A2_col=A2_col,
                  B1_col=B1_col,
    save_path="../../figures/from_260430/"   # folder -> auto filename
)

kde_all_pcc(res, 
            A1_col=A1_col, 
            B1_col=B1_col, 
            save_path="../../figures/from_260430/")

Saved: ../../figures/from_260312/pcc_boxplot_R0_vs_max_time_by_sigma.png
Saved: ../../figures/from_260312/pcc_violin_R0_vs_max_time_by_sigma.png
Saved: ../../figures/from_260312/pcc_kde_all_R0_vs_max_time.png


'../../figures/from_260312/pcc_kde_all_R0_vs_max_time.png'